In [4]:
from pathlib import Path
from collections import Counter

import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchaudio.datasets import IEMOCAP
from torch.utils.data import DataLoader, Subset

import os
import scipy.io.wavfile
import numpy as np

# sample_rate, signal = scipy.io.wavfile.read('example.wav')

In [5]:
print("PyTorch :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())
print("CUDA version (build) :", torch.version.cuda)   # ex) '12.1' or None
print("cuDNN available :", torch.backends.cudnn.is_available())
print("cuDNN version   :", torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None)
print("# of GPUs :", torch.cuda.device_count())

PyTorch : 2.10.0.dev20251001+cu126
CUDA available : True
CUDA version (build) : 12.6
cuDNN available : True
cuDNN version   : 91002
# of GPUs : 1


In [7]:
#데이터셋 확인
path = os.getcwd()
print(path)

root = Path(path + "/IEMOCAP_full_release")
ds = IEMOCAP(str(root), sessions=(1,2,3,4,5))  # 필요한 세션만 골라도 됨

# 샘플 확인
waveform, sr, fname, label, speaker = ds[0]
print(sr, fname, label, speaker)

labels = Counter()
srs = Counter()
for i in range(min(len(ds), 2000)):  # 처음 2000개만 훑기
    wf, sr, _, label, _ = ds[i]
    labels[label] += 1
    srs[sr] += 1

print("Label dist:", labels)
print("Sample rates:", srs)

c:\Users\USER\Desktop\2025-2\세미나 논문\IEMOCAP_code


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6 and 7.
          2. The PyTorch version (2.10.0.dev20251001+cu126) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.
        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 7: Could not load this library: C:\Users\USER\AppData\Local\Programs\Python\Python310\Lib\site-packages\torchcodec\libtorchcodec_core7.dll
FFmpeg version 6: Could not load this library: C:\Users\USER\AppData\Local\Programs\Python\Python310\Lib\site-packages\torchcodec\libtorchcodec_core6.dll
FFmpeg version 5: Could not load this library: C:\U

In [8]:
from pathlib import Path
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
from fractions import Fraction

def load_wav_safely(path, target_sr=None, mono=True):
    # 1) 읽기
    sr, sig = wavfile.read(str(path))  # sig: np.ndarray

    # 2) float32 [-1,1] 정규화
    if sig.dtype == np.int16:
        sig = sig.astype(np.float32) / 32768.0
    elif sig.dtype == np.int32:
        sig = sig.astype(np.float32) / 2147483648.0
    elif sig.dtype == np.uint8:
        sig = (sig.astype(np.float32) - 128.0) / 128.0
    else:
        sig = sig.astype(np.float32)  # float형이면 그대로

    # 3) 모노 변환(스테레오일 때 평균)
    if mono and sig.ndim == 2:
        sig = sig.mean(axis=1)

    # 4) 리샘플(선택)
    if target_sr is not None and sr != target_sr:
        # 정수 비율 근사로 poly resample
        frac = Fraction(target_sr, sr).limit_denominator()
        sig = resample_poly(sig, frac.numerator, frac.denominator).astype(np.float32)
        sr = target_sr

    return sr, sig  # (int, np.ndarray[T,])

# ====== 사용 예시 ======
root = Path(r"C:\Users\USER\Desktop\2025-2\세미나 논문\IEMOCAP_code\IEMOCAP_full_release\IEMOCAP\Session1\dialog\wav")
wav_paths = sorted(root.rglob("*.wav"))  # 하위 폴더까지 모두

print(f"found {len(wav_paths)} wavs")
for p in wav_paths:
    sr, sig = load_wav_safely(p, target_sr=16000, mono=True)
    dur = len(sig) / sr
    print(p.name, f"{sr} Hz", f"{dur:.2f} sec")

found 28 wavs
Ses01F_impro01.wav 16000 Hz 133.63 sec
Ses01F_impro02.wav 16000 Hz 220.28 sec
Ses01F_impro03.wav 16000 Hz 131.11 sec
Ses01F_impro04.wav 16000 Hz 222.91 sec
Ses01F_impro05.wav 16000 Hz 228.73 sec
Ses01F_impro06.wav 16000 Hz 333.00 sec
Ses01F_impro07.wav 16000 Hz 149.39 sec
Ses01F_script01_1.wav 16000 Hz 476.94 sec
Ses01F_script01_2.wav 16000 Hz 162.02 sec
Ses01F_script01_3.wav 16000 Hz 474.01 sec
Ses01F_script02_1.wav 16000 Hz 350.25 sec
Ses01F_script02_2.wav 16000 Hz 489.14 sec
Ses01F_script03_1.wav 16000 Hz 314.05 sec
Ses01F_script03_2.wav 16000 Hz 325.18 sec
Ses01M_impro01.wav 16000 Hz 172.76 sec
Ses01M_impro02.wav 16000 Hz 229.03 sec
Ses01M_impro03.wav 16000 Hz 161.58 sec
Ses01M_impro04.wav 16000 Hz 193.24 sec
Ses01M_impro05.wav 16000 Hz 206.21 sec
Ses01M_impro06.wav 16000 Hz 253.18 sec
Ses01M_impro07.wav 16000 Hz 212.19 sec
Ses01M_script01_1.wav 16000 Hz 471.50 sec
Ses01M_script01_2.wav 16000 Hz 181.63 sec
Ses01M_script01_3.wav 16000 Hz 539.39 sec
Ses01M_script02_1.wa

In [9]:
# pre_emphasis : 고주파 성분의 에너지를 올려주는 전처리 과정
#Signal-to-Noise Ratio(SNR) 개선

'''sample_rate, signal = scipy.io.wavfile.read('example.wav')
pre_emphasis = 0.97
emphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])

# 원시 음성을 짧은 구간(25ms)으로 자르되 일부 구간(10ms)은 겹치게 처리
frame_size = 0.025
frame_stride = 0.01
frame_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate

signal_length = len(emphasized_signal)
frame_length = int(round(frame_length))
frame_step = int(round(frame_step))

num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))
pad_signal_length = num_frames * frame_step + frame_length

z = np.zeros((pad_signal_length - signal_length))
pad_signal = np.append(emphasized_signal, z)
indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
          np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
frames = pad_signal[indices.astype(np.int32, copy=False)]'''


"sample_rate, signal = scipy.io.wavfile.read('example.wav')\npre_emphasis = 0.97\nemphasized_signal = np.append(signal[0], signal[1:] - pre_emphasis * signal[:-1])\n\n# 원시 음성을 짧은 구간(25ms)으로 자르되 일부 구간(10ms)은 겹치게 처리\nframe_size = 0.025\nframe_stride = 0.01\nframe_length, frame_step = frame_size * sample_rate, frame_stride * sample_rate\n\nsignal_length = len(emphasized_signal)\nframe_length = int(round(frame_length))\nframe_step = int(round(frame_step))\n\nnum_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))\npad_signal_length = num_frames * frame_step + frame_length\n\nz = np.zeros((pad_signal_length - signal_length))\npad_signal = np.append(emphasized_signal, z)\nindices = np.tile(np.arange(0, frame_length), (num_frames, 1)) +           np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T\nframes = pad_signal[indices.astype(np.int32, copy=False)]"

In [10]:
sr = 16000
mfcc = torchaudio.transforms.MFCC(sample_rate=sr, n_mfcc=40, melkwargs={"n_mels":64})
d = torchaudio.transforms.ComputeDeltas()
dd = torchaudio.transforms.ComputeDeltas(win_length=5)

def mfcc_stack(waveform, sr_in):
    if sr_in != sr:
        waveform = torchaudio.functional.resample(waveform, sr_in, sr)
    x = mfcc(waveform)            # (ch=1, n_mfcc, T)
    dx = d(x); ddx = dd(dx)
    feat = torch.cat([x, dx, ddx], dim=1)  # (1, 40*3, T)
    return feat.squeeze(0)        # (120, T)

In [11]:
class PreEmphasis(nn.Module):
    def __init__(self, a=0.97): super().__init__(); self.a=a
    def forward(self, x):  # (B,1,T)
        return torch.cat([x[..., :1], x[..., 1:] - self.a*x[..., :-1]], dim=-1)

class LogMel(nn.Module):
    def __init__(self, sr=16000, n_mels=64, win_ms=25, hop_ms=10, preemph=0.97):
        super().__init__()
        n_fft     = int(round(sr*win_ms/1000))
        hop_len   = int(round(sr*hop_ms/1000))
        self.pre  = PreEmphasis(preemph) if preemph else nn.Identity()
        self.mel  = torchaudio.transforms.MelSpectrogram(sr, n_fft=n_fft,
                    win_length=n_fft, hop_length=hop_len, n_mels=n_mels, power=2.0)
        self.todb = torchaudio.transforms.AmplitudeToDB(stype="power")
    def forward(self, wav):  # (B,1,T)
        x = self.pre(wav)
        M = self.mel(x).clamp_min(1e-10)
        return self.todb(M)  # (B, n_mels, F)

# --- Segmentation on waveform ---
def segment_waveform(wav, sr=16000, seg_sec=2.0, overlap=0.5):
    # wav: (B,1,T)
    B, _, T = wav.shape
    L = int(round(seg_sec*sr)); H = int(round(L*(1-overlap)))
    if T <= L:  # pad if too short
        wav = F.pad(wav, (0, L-T))
        T = wav.shape[-1]
    starts = torch.arange(0, T - L + 1, H, device=wav.device)
    segs = torch.stack([wav[..., s:s+L] for s in starts], dim=1)  # (B, S, 1, L)
    return segs  # S: #segments

# --- CNN Feature Encoder: (1, n_mels, F) -> (d_model) per segment ---
class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d((2,2)),
            nn.Conv2d(32,64,3, padding=1),  nn.ReLU(), nn.MaxPool2d((2,2)),
            nn.Conv2d(64,128,3,padding=1),  nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1)), nn.Flatten(),
            nn.Linear(128, d_model)
        )
    def forward(self, mel):  # (B,S, n_mels, F)
        B,S,N,Ft = mel.shape
        x = mel.unsqueeze(2)                # (B,S,1,n_mels,F)
        x = x.reshape(B*S, 1, N, Ft)        # (B*S,1,n_mels,F)
        z = self.net(x)                     # (B*S, d_model)
        return z.view(B, S, -1)             # (B,S,d_model)
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2)*(-torch.log(torch.tensor(10000.0))/d_model))
        pe[:,0::2] = torch.sin(pos*div); pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe)  # (max_len, d_model)
    def forward(self, x):  # (B,S,d)
        S = x.size(1)
        return x + self.pe[:S].unsqueeze(0)

class AudioTransEncoder(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_layers=4, dim_ff=512, num_classes=None):
        super().__init__()
        self.pos = PositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout=0.1, batch_first=True)
        self.tr  = nn.TransformerEncoder(enc_layer, num_layers)
        self.cls = nn.Parameter(torch.zeros(1,1,d_model))  # [CLS]
        self.head = nn.Linear(d_model, num_classes) if num_classes else nn.Identity()
    def forward(self, E, mask=None):  # E: (B,S,d)
        B = E.size(0)
        cls = self.cls.expand(B, -1, -1)
        X = torch.cat([cls, E], dim=1)        # (B,S+1,d)
        X = self.pos(X)
        Z = self.tr(X, src_key_padding_mask=mask)  # mask: (B,S+1) optional
        H = Z[:,0]                               # [CLS]
        return H, self.head(H)

# --- Full model wrapper ---
class AudioSER(nn.Module):
    def __init__(self, sr=16000, n_mels=64, d_model=256, nhead=4, layers=4, dim_ff=512, num_classes=6):
        super().__init__()
        self.front = LogMel(sr, n_mels)
        self.cnn   = CNNEncoder(d_model)
        self.tr    = AudioTransEncoder(d_model, nhead, layers, dim_ff, num_classes)
        self.sr = sr
    def forward(self, wav):  # (B,1,T)
        segs = segment_waveform(wav, self.sr, seg_sec=2.0, overlap=0.5)   # (B,S,1,L)
        # front per segment
        B,S,_,L = segs.shape
        mel = self.front(segs.reshape(B*S,1,L))            # (B*S, n_mels, F)
        E   = self.cnn(mel.reshape(B, S, mel.size(1), mel.size(2)))  # (B,S,d)
        H, logits = self.tr(E)  # H: (B,d)
        return H, logits

In [12]:
try:
    torchaudio.set_audio_backend("sox_io")
except:
    torchaudio.set_audio_backend("soundfile")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 6  
model = AudioSER(sr=16000, n_mels=64, d_model=256, nhead=4, layers=4,
                 dim_ff=512, num_classes=num_classes).to(device)
model.eval()

AttributeError: module 'torchaudio' has no attribute 'set_audio_backend'

In [ ]:
print("GPU:", torch.cuda.get_device_name(0))
print("CC :", torch.cuda.get_device_capability(0))  # 예: (6, 1) → Pascal
print("torch.version.cuda:", torch.version.cuda)

GPU: NVIDIA GeForce RTX 5060
CC : (12, 0)
torch.version.cuda: 12.4


In [ ]:
map_label = {"ang":0, "fru":1, "hap":2, "neu":3, "sad":4}
def norm_label(lab):
    return "hap" if lab == "exc" else lab

# 간단 collate: 배치=1로 먼저 검증(변길이 세그먼트 패딩 고민 無)
def collate_b1(batch):
    wav, sr, fname, label, spk = batch[0]
    y = map_label[norm_label(label)]
    return wav.unsqueeze(0), torch.tensor([y])  # (B=1,1,T), (1,)

loader = DataLoader(ds, batch_size=1, shuffle=True, collate_fn=collate_b1)

# 학습 스켈레톤
model.train()
optim = torch.optim.Adam(model.parameters(), lr=3e-4)
crit  = nn.CrossEntropyLoss()

for step, (wav, y) in enumerate(loader, 1):
    wav = wav.to(device)   # (1,1,T)
    y   = y.to(device)     # (1,)
    H, logits = model(wav) # (1,d), (1,C)
    loss = crit(logits, y)
    optim.zero_grad(); loss.backward(); optim.step()

    if step % 100 == 0:
        pred = logits.argmax(-1).item()
        print(f"[{step}] loss={loss.item():.4f}  y={y.item()} pred={pred}")
    if step == 1000:   # 데모 목적: 1000 스텝 정도만
        break

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased")

def bert_sentence_embedding(text: str):
    t = tok(text, return_tensors="pt", truncation=True, max_length=128)
    out = bert(**t)
    # [CLS] 풀링(또는 last hidden states 평균)
    cls = out.last_hidden_state[:,0]   # (1, hidden_size=768)
    return cls.squeeze(0)              # (768,)

c:\Users\USER\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
